# 05. First-Order Logic

Notebook 03 sempat menyinggung batasan propositional logic: `W12` cuma
nama atomik, bukan pemanggilan predicate `W(1, 2)` -- dua kotak berbeda
yang kebetulan bertetangga sama sekali tidak dikenali sebagai punya
struktur yang sama. **First-order logic** (FOL) menutup celah itu: dunia
direpresentasikan lewat *objects*, *relations*, dan *functions*, bukan
cuma sekumpulan proposisi lepas.

Setelah menyelesaikan notebook ini, Anda diharapkan mampu:

1. menjelaskan kenapa FOL diperlukan lewat kasus yang tidak bisa
   direpresentasikan ringkas di propositional logic;
2. membedakan constant, variable, predicate, dan function dalam syntax
   FOL, dan membangunnya lewat `Expr`/`expr()`;
3. membedakan universal ($\forall$) dan existential ($\exists$)
   quantifier, termasuk dua kesalahan umum memasangkannya dengan
   connective; dan
4. menerjemahkan kalimat natural language ke FOL, termasuk membangun
   sebuah knowledge base FOL yang dipakai untuk inferensi di Notebook 06.

## Setup

Jalankan sel di bawah ini sekali di awal, sebelum sel mana pun yang lain.

Sel ini memasang dependensi yang diperlukan, mencari folder yang berisi
`logic.py` dan `utils.py`, lalu mengimpornya. Kalau notebook dibuka lewat Google
Colab, repo akan di-clone otomatis. Tidak ada yang perlu diubah di sini.

Environment sudah siap kalau baris terakhir output mencetak
`Check       : tt_entails(P & Q, Q) = True`.

In [1]:
# =============================================================================
# Standard setup cell.
# Run this once, before any other cell in this notebook.
# =============================================================================
import importlib.util
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kcv-if/Modul-Praktikum-KK-RKA-25.git"
ON_COLAB = "google.colab" in sys.modules


def ensure_dependencies():
    """Install only the packages this module actually uses."""
    required = {
        "networkx": "networkx",
        "numpy": "numpy",
        "pandas": "pandas",
        "matplotlib": "matplotlib",
        "ipywidgets": "ipywidgets",
        "PIL": "pillow",
        "pygments": "pygments",
    }
    missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", ", ".join(missing))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)


def find_environment(start):
    """Locate the folder that holds logic.py and utils.py, searching upward."""
    for root in [start, *start.parents]:
        for candidate in sorted(root.rglob("logic.py")):
            if (candidate.parent / "utils.py").exists():
                return candidate.parent
        if (root / ".git").exists():
            break
    return None


ensure_dependencies()

start_dir = Path.cwd()
if ON_COLAB:
    clone_dir = Path("Modul-Praktikum-KK-RKA-25")
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)], check=True)
    start_dir = clone_dir

ENV_DIR = find_environment(start_dir)
if ENV_DIR is None:
    raise RuntimeError(
        "Environment folder not found. Make sure this notebook is opened from "
        "inside the Modul-Praktikum-KK-RKA-25 repository."
    )
if str(ENV_DIR) not in sys.path:
    sys.path.insert(0, str(ENV_DIR))

import itertools
import warnings

import pandas as pd

# qpsolvers is only used by the SVM code, which this module never touches.
warnings.filterwarnings("ignore", message="no QP solver found")

from logic import *
from notebook import psource
from utils import *

print("Environment :", ENV_DIR)
print("Python      :", sys.version.split()[0])
print("Check       : tt_entails(P & Q, Q) =", tt_entails(expr("P & Q"), expr("Q")))

Environment : C:\Users\cathl\Kuliah\KCV\KK\Modul-Praktikum-KK-RKA-25\logics\praktikum\environment
Python      : 3.14.2
Check       : tt_entails(P & Q, Q) = True


---
# 5.1 Kenapa First-Order Logic?

## Penjelasan

Propositional logic mengasumsikan dunia sebagai sekumpulan **fact**
lepas. Untuk menyatakan sesuatu tentang banyak object sekaligus, misalnya
"semua mahasiswa mengambil minimal satu mata kuliah," propositional logic
terpaksa menulis satu proposisi terpisah untuk **setiap** mahasiswa --
tidak ada cara menggeneralisasi lewat "semua x" atau "ada x."

First-order logic, seperti natural language, mengasumsikan dunia berisi:

- **objects**: orang, angka, warna, pertandingan, negara, dan sebagainya;
- **relations**: merah, prima, saudara-dari, lebih-besar-dari, dan
  sebagainya -- properti dan hubungan antar object;
- **functions**: ayah-dari, teman-terdekat, satu-lebih-dari, dan
  sebagainya -- pemetaan dari object ke object lain.

Ingat lagi batasan `W12` di Notebook 03: propositional logic tidak tahu
bahwa `W12` dan `W13` berhubungan sama sekali, keduanya cuma dua nama
yang kebetulan mirip. FOL mengatasi ini lewat **predicate**: `W(1, 2)`
dan `W(1, 3)` sama-sama pemanggilan predicate `W`, cuma beda argumen --
dua kotak berbeda dari relasi yang sama, dikenali sebagai punya struktur
yang sama.

Studi kasus utama sepanjang Notebook 05 dan 06 bukan Wumpus World lagi,
tapi kasus "Colonel West" dari slide: siapa yang pantas disebut kriminal
berdasarkan aturan penjualan senjata. Notebook ini membangun
representasinya; Notebook 06 membuktikannya.

## Contoh penerapan

Bandingkan langsung: symbol atomik ala Notebook 03 lawan predicate call
ala FOL, keduanya sama-sama objek `Expr`, tapi struktur `.args`-nya
beda.

In [2]:
# Notebook 03: W12 atomik, tidak ada struktur di dalamnya.
w12 = Symbol('W12')
w13 = Symbol('W13')

# Notebook 05: W(1, 2) adalah pemanggilan predicate W dengan argumen 1, 2.
w_1_2 = expr('W(1, 2)')
w_1_3 = expr('W(1, 3)')

print("W12    -> op:", repr(w12.op), " args:", w12.args)
print("W(1,2) -> op:", repr(w_1_2.op), " args:", w_1_2.args)
print()
print("W12 dan W13 berbagi predicate yang sama?    ", w12.op == w13.op)
print("W(1,2) dan W(1,3) berbagi predicate yang sama?", w_1_2.op == w_1_3.op)

W12    -> op: 'W12'  args: ()
W(1,2) -> op: 'W'  args: (1, 2)

W12 dan W13 berbagi predicate yang sama?     False
W(1,2) dan W(1,3) berbagi predicate yang sama? True


---
# 5.2 Syntax: Constant, Predicate, Function, Variable

## Penjelasan

Syntax FOL mengenal empat elemen dasar:

- **Constant**: menamai satu object tertentu. Contoh: `KingJohn`, `2`,
  `NUS`.
- **Predicate**: menamai relasi atau properti, diterapkan ke satu atau
  lebih term, hasilnya sentence (bisa benar/salah). Contoh: `Brother(x,
  y)`, `>(x, y)`.
- **Function**: memetakan term ke term **lain** -- bukan ke benar/salah.
  Contoh: `LeftLegOf(x)`, `FatherOf(x)`. Bedanya dengan predicate ini
  penting: `Brother(KingJohn, Richard)` adalah sentence yang bisa
  dinilai benar/salah, tapi `LeftLegOf(Richard)` cuma menunjuk ke sebuah
  object (kaki kiri Richard), bukan pernyataan yang bisa benar/salah.
- **Variable**: placeholder untuk object yang belum ditentukan. Konvensi
  penulisannya diawali huruf kecil (`x`, `y`, `z`), sedangkan constant,
  predicate, dan function diawali huruf besar.

Konvensi huruf besar/kecil ini bukan cuma gaya penulisan: `logic.py`
memakainya untuk **membedakan** variable dari constant secara otomatis
lewat `is_var_symbol`, tanpa perlu deklarasi terpisah. Ini bukan detail
sepele -- Notebook 06 (unifikasi) bergantung penuh ke konvensi ini untuk
tahu simbol mana yang boleh disubstitusi dan mana yang tidak.

Term dan atomic sentence disusun begini:

$$\text{Term} \to \text{Function}(\text{Term}, \dots) \mid \text{Constant}
\mid \text{Variable}$$
$$\text{AtomicSentence} \to \text{Predicate}(\text{Term}, \dots) \mid
\text{Term} = \text{Term}$$

Contoh dari buku: `Brother(KingJohn, RichardTheLionheart)` adalah atomic
sentence. `>(Length(LeftLegOf(Richard)), Length(LeftLegOf(KingJohn)))`
lebih rumit: predicate `>` diterapkan ke dua term, dan masing-masing term
itu sendiri berupa function yang bersarang tiga tingkat.

`expr()` mengeval string-nya sebagai kode Python (lihat source-nya
sendiri di 5.1 kalau lupa), jadi predicate `>` dari buku ini satu-satunya
yang tidak bisa diketik apa adanya: `>` adalah operator Python, bukan
identifier yang sah untuk memulai sebuah pemanggilan. Di kode di bawah,
predicate itu diberi nama `GT` sebagai gantinya -- strukturnya, bukan
namanya, yang jadi inti contoh ini.

## Contoh penerapan

Cek konvensi huruf besar/kecil langsung lewat `is_var_symbol`, lalu
bangun kedua contoh dari buku di atas.

In [3]:
psource(is_var_symbol, is_variable)

print("'KingJohn' variable?", is_var_symbol('KingJohn'))
print("'x' variable?       ", is_var_symbol('x'))

'KingJohn' variable? False
'x' variable?        True


In [4]:
brother = expr('Brother(KingJohn, RichardTheLionheart)')
print(brother)
print("op  :", brother.op)
print("args:", brother.args)

Brother(KingJohn, RichardTheLionheart)
op  : Brother
args: (KingJohn, RichardTheLionheart)


In [5]:
# Predicate '>' dari buku ditulis GT di sini (lihat Penjelasan), diterapkan
# ke dua term, masing-masing term berupa Function yang bersarang:
# LeftLegOf(...) di dalam Length(...).
leg_length_cmp = expr('GT(Length(LeftLegOf(Richard)), Length(LeftLegOf(KingJohn)))')
print(leg_length_cmp)

first_term = leg_length_cmp.args[0]
print("Term pertama:", first_term, "-- op:", first_term.op)

GT(Length(LeftLegOf(Richard)), Length(LeftLegOf(KingJohn)))
Term pertama: Length(LeftLegOf(Richard)) -- op: Length


`leg_length_cmp` sendiri adalah atomic sentence (predicate `GT`), tapi
`first_term` di dalamnya cuma sebuah term (function `Length`), bukan
sentence -- tidak masuk akal menanyakan apakah `Length(LeftLegOf(Richard))`
benar atau salah, beda dengan `leg_length_cmp` yang punya nilai
kebenaran.

---
# 5.3 Quantifier

## Penjelasan

**Universal quantifier** ($\forall$): $\forall x\ P(x)$ benar pada suatu
model jika dan hanya jika $P$ benar untuk **setiap** object di model itu.
Setara dengan conjunction raksasa atas seluruh object:

$$\forall x\ (\text{At}(x, \text{NUS}) \Rightarrow \text{Smart}(x))
\equiv \text{At}(\text{KingJohn}, \text{NUS}) \Rightarrow
\text{Smart}(\text{KingJohn}) \ \land\ \text{At}(\text{Richard},
\text{NUS}) \Rightarrow \text{Smart}(\text{Richard}) \ \land\ \dots$$

**Existential quantifier** ($\exists$): $\exists x\ P(x)$ benar jika $P$
benar untuk **sedikitnya satu** object. Setara dengan disjunction
raksasa:

$$\exists x\ (\text{At}(x, \text{NUS}) \land \text{Smart}(x)) \equiv
\text{At}(\text{KingJohn}, \text{NUS}) \land \text{Smart}(\text{KingJohn})
\ \lor\ \text{At}(\text{Richard}, \text{NUS}) \land
\text{Smart}(\text{Richard}) \ \lor\ \dots$$

**Dua kesalahan pemasangan yang paling umum:**

- Memakai $\land$ dengan $\forall$: $\forall x\ \text{At}(x, \text{NUS})
  \land \text{Smart}(x)$ artinya "semua orang ada di NUS **dan** semua
  orang pintar" -- klaim yang jauh lebih kuat dan hampir pasti salah,
  bukan "semua yang di NUS itu pintar."
- Memakai $\Rightarrow$ dengan $\exists$: $\exists x\ \text{At}(x,
  \text{NUS}) \Rightarrow \text{Smart}(x)$ bernilai benar **selama ada
  satu saja** orang yang tidak di NUS -- implikasi dengan premise salah
  otomatis benar (Notebook 03, 3.4), jadi sentence ini nyaris selalu
  benar tanpa menyatakan apa-apa yang berguna soal NUS.

Aturan praktis: $\forall$ berpasangan dengan $\Rightarrow$, $\exists$
berpasangan dengan $\land$.

**Urutan quantifier penting.** $\exists x\ \forall y\ \text{Loves}(x, y)$
("ada satu orang yang mencintai semua orang") **tidak sama** dengan
$\forall y\ \exists x\ \text{Loves}(x, y)$ ("setiap orang dicintai oleh
seseorang, tapi belum tentu orang yang sama"). Quantifier sejenis boleh
ditukar urutannya bebas ($\forall x \forall y \equiv \forall y \forall
x$, begitu juga $\exists$), tapi quantifier berbeda jenis tidak.

**Quantifier duality**: $\forall x\ P \equiv \neg \exists x\ \neg P$, dan
$\exists x\ P \equiv \neg \forall x\ \neg P$ -- pola yang sama dengan De
Morgan (Notebook 03, Soal 3), cuma untuk quantifier.

**Equality** ($=$): `term1 = term2` benar jika dan hanya jika keduanya
menunjuk ke object yang sama. Dipakai misalnya mendefinisikan `Sibling`
lewat `Parent` yang sama: dua orang bersaudara jika mereka bukan orang
yang sama ($\neg(x = y)$) dan berbagi sedikitnya satu induk.

**Catatan penting, sekaligus batasan.** `logic.py` **tidak** punya
syntax untuk $\forall$ maupun $\exists$ -- tidak ada simbolnya sama
sekali di `Expr`/`expr()`. Ini bukan kekurangan yang kebetulan terlewat:
satu-satunya inferensi FOL yang dipakai di Notebook 06 (forward dan
backward chaining) cuma butuh KB berisi **definite clause**, generalisasi
dari Notebook 04 -- dan di situ, setiap variable yang muncul di sebuah
rule otomatis dianggap "untuk semua nilai variable ini" (implicit
$\forall$), tanpa perlu ditulis. Konsekuensinya: **tidak ada cara
menulis $\exists$ sama sekali** di `logic.py` -- ini batasan ekspresif
nyata dari pembatasan definite-clause yang sama yang membuat
forward/backward chaining efisien di Notebook 04.

## Contoh penerapan

`variables()` mengambil semua variable dalam sebuah sentence -- inilah
"implicit $\forall$" yang dimaksud di atas, murni dari konvensi
huruf-kecil, tanpa simbol quantifier sama sekali.

In [6]:
psource(variables)

rule = expr('(Student(x) & Takes(x, AI)) ==> Eligible(x, AILab)')
print(rule)
print("Variabel (otomatis dianggap 'untuk semua'):", variables(rule))

((Student(x) & Takes(x, AI)) ==> Eligible(x, AILab))
Variabel (otomatis dianggap 'untuk semua'): {x}


`x` terdeteksi murni karena huruf awalnya kecil (`is_var_symbol`, 5.2),
tanpa satu pun simbol $\forall$ ditulis di mana pun. `AI` dan `AILab`
tidak ikut terdeteksi sebagai variable karena diawali huruf besar --
keduanya constant, bukan sesuatu yang mau digeneralisasi.

---
# 5.4 Translate Natural Language -> FOL: KB Colonel West

## Penjelasan

Skenario dari slide, dipakai sebagai studi kasus utama Notebook 05 dan
06:

> The law says that it is a crime for an American to sell weapons to
> hostile nations. The country Nono, an enemy of America, has some
> missiles, and all of its missiles were sold to it by Colonel West, who
> is American.

Target akhirnya (dibuktikan di Notebook 06): **Colonel West is a
criminal**. Notebook ini cuma menerjemahkan tiap kalimat ke FOL, belum
membuktikan apa pun.

| Kalimat | FOL |
|---|---|
| adalah kejahatan bagi orang Amerika menjual senjata ke negara musuh | $\text{American}(x) \land \text{Weapon}(y) \land \text{Sells}(x, y, z) \land \text{Hostile}(z) \Rightarrow \text{Criminal}(x)$ |
| Nono punya rudal (missile) | $\exists x\ \text{Owns}(\text{Nono}, x) \land \text{Missile}(x)$ |
| seluruh rudal Nono dijual oleh Colonel West | $\text{Missile}(x) \land \text{Owns}(\text{Nono}, x) \Rightarrow \text{Sells}(\text{West}, x, \text{Nono})$ |
| rudal adalah senjata | $\text{Missile}(x) \Rightarrow \text{Weapon}(x)$ |
| musuh Amerika terhitung "hostile" | $\text{Enemy}(x, \text{America}) \Rightarrow \text{Hostile}(x)$ |
| West orang Amerika | $\text{American}(\text{West})$ |
| Nono musuh Amerika | $\text{Enemy}(\text{Nono}, \text{America})$ |

Baris kedua sedikit berbeda dari yang lain: dia pakai $\exists x$, bukan
$\Rightarrow$ berbentuk rule. Dalam praktiknya baris ini diterjemahkan
jadi dua fact konkret, `Owns(Nono, M1)` dan `Missile(M1)`, dengan `M1`
sebagai nama rudal tertentu -- persis pola "ada minimal satu, beri nama,
jadikan fact" yang disinggung sekilas di 5.3 lewat konsep instantiation
(dibahas lebih jauh di Notebook 06, 6.1).

Rule pertama, aturan kejahatannya sendiri, adalah rule dengan **empat**
premise sekaligus: `American(x)`, `Weapon(y)`, `Sells(x, y, z)`, dan
`Hostile(z)` semuanya harus terpenuhi (dengan `x`, `y`, `z` konsisten)
sebelum `Criminal(x)` boleh disimpulkan.

## Contoh penerapan

Bangun seluruh KB di atas sebagai `FolKB`. KB yang sama persis sudah
tersedia di `logic.py` sebagai `crime_kb` -- setelah membangunnya
sendiri, bandingkan keduanya.

In [7]:
psource(FolKB)

In [8]:
crime_rule = expr('(American(x) & Weapon(y) & Sells(x, y, z) & Hostile(z)) ==> Criminal(x)')
sells_missile = expr('(Missile(x) & Owns(Nono, x)) ==> Sells(West, x, Nono)')
missile_is_weapon = expr('Missile(x) ==> Weapon(x)')
enemy_is_hostile = expr('Enemy(x, America) ==> Hostile(x)')

facts = [
    expr('Owns(Nono, M1)'),
    expr('Missile(M1)'),
    expr('American(West)'),
    expr('Enemy(Nono, America)'),
]

my_crime_kb = FolKB([crime_rule, sells_missile, missile_is_weapon, enemy_is_hostile] + facts)
for clause in my_crime_kb.clauses:
    print(clause)

((((American(x) & Weapon(y)) & Sells(x, y, z)) & Hostile(z)) ==> Criminal(x))
((Missile(x) & Owns(Nono, x)) ==> Sells(West, x, Nono))
(Missile(x) ==> Weapon(x))
(Enemy(x, America) ==> Hostile(x))
Owns(Nono, M1)
Missile(M1)
American(West)
Enemy(Nono, America)


In [9]:
# Persis kasus yang sama sudah tersedia di logic.py sebagai crime_kb.
for clause in crime_kb.clauses:
    print(clause)

print()
print("Jumlah clause sama?", len(my_crime_kb.clauses) == len(crime_kb.clauses))

((((American(x) & Weapon(y)) & Sells(x, y, z)) & Hostile(z)) ==> Criminal(x))
Owns(Nono, M1)
Missile(M1)
((Missile(x) & Owns(Nono, x)) ==> Sells(West, x, Nono))
(Missile(x) ==> Weapon(x))
(Enemy(x, America) ==> Hostile(x))
American(West)
Enemy(Nono, America)

Jumlah clause sama? True


Isinya sama (urutannya boleh beda, itu tidak masalah -- `FolKB` cuma
menyimpan clause sebagai list). Notebook 06 memakai `crime_kb` langsung,
tanpa membangunnya ulang, persis karena sudah dibangun di sini.

---
# Latihan Soal

## Soal 1

Formula berikut bermaksud menyatakan "semua mahasiswa itu pintar," tapi
salah menuliskannya:

$$\forall x\ \text{Student}(x) \land \text{Smart}(x)$$

Jelaskan apa yang sebenarnya dinyatakan formula ini (bukan cuma "salah"),
lalu tulis versi yang benar.

<details>
<summary>Klik untuk melihat hint</summary>

Coba "masukkan" satu object yang **bukan** mahasiswa ke formula ini,
misalnya sebuah kursi -- ingat 5.3, $\forall x\ P(x) \land Q(x)$ setara
dengan conjunction raksasa atas *semua* object. Apa formula ini tetap
mengklaim sesuatu tentang si kursi? Bandingkan dengan versi yang
pasangannya $\Rightarrow$, lalu cek lagi kenapa implikasi tidak
memaksakan klaim apa pun ke object yang premise-nya salah (Notebook 03
Soal 4).

</details>

## Soal 2

Terjemahkan dua kalimat berikut ke FOL, bangun sebagai `Expr` lewat
`expr()`, lalu cetak `.op` dan `.args` masing-masing untuk memastikan
strukturnya sesuai:

1. "Semua mahasiswa yang mengambil AI dan sudah lulus Pemrograman boleh
   ikut praktikum AI."
2. "Ada mahasiswa yang mengambil AI."

<details>
<summary>Klik untuk melihat hint</summary>

Kalimat 1: ada berapa syarat yang harus dipenuhi sekaligus sebelum
"boleh ikut praktikum AI"? Itu jumlah premise yang perlu di-AND sebelum
`==>`.

Kalimat 2 pakai existential quantifier di buku ("ada mahasiswa..."),
tapi ingat catatan di 5.3: `logic.py` tidak punya syntax untuk $\exists$
sama sekali. 5.4 menghadapi situasi yang sama persis untuk "Nono punya
rudal" -- lihat lagi bagaimana 5.4 menerjemahkannya jadi fact konkret,
lalu pakai pola yang sama di sini.

</details>

## Soal 3

Buat **satu contoh kalimat nyata sendiri** (bukan dari materi, bukan
"Loves") yang menunjukkan perbedaan makna antara $\exists x\ \forall y\
P(x, y)$ dan $\forall y\ \exists x\ P(x, y)$. Tulis kedua versi FOL-nya,
dan jelaskan dengan kata-kata kenapa keduanya menyatakan hal yang
berbeda.

<details>
<summary>Klik untuk melihat hint</summary>

Pikirkan predicate biner "x melakukan sesuatu untuk/ke y" di dunia
nyata sekitarmu (bukan `Loves`, bukan dari materi -- boleh apa saja:
relasi mengajar, melayani, mengawasi, dan sejenisnya).

Bayangkan dulu $\exists x\ \forall y\ P(x, y)$ benar: apa konsekuensinya
buat **satu** object $x$ tertentu -- dia harus berhubungan dengan siapa
saja? Sekarang bayangkan yang benar cuma $\forall y\ \exists x\ P(x,
y)$: apa $x$-nya wajib **sama** untuk setiap $y$, atau boleh
berbeda-beda? Dari situ, mana klaim yang lebih "kuat" (lebih susah
dipenuhi)?

</details>